In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col, count, when, isnan

catalog_name = 'automobilerepair'

In [0]:
df = spark.read.table("automobilerepair.bronze.stg_estimate")

In [0]:
display(df.limit(5))

In [0]:
row_count = df.count()
row_count

In [0]:
print("\nDUPLICATE ANALYSIS")
duplicate_estimate_ids = df.groupBy("estimate_id").count().filter(col("count") > 1)
print(f"Duplicate estimate_ids: {duplicate_estimate_ids.count()}")

In [0]:
print("\nNULL VALUE ANALYSIS")

null_counts = df.select([ count(when(col(c).isNull(), c)).alias(c) for c in df.columns ])
print("Null counts by column:")
display(null_counts)

In [0]:
# Handle null values in 'estimate_amount' by imputing with median
median_estimate_amount = df.approxQuantile('estimate_amount', [0.5], 0.01)[0]
df = df.withColumn(
    'estimate_amount',
    when(col('estimate_amount').isNull(), median_estimate_amount).otherwise(col('estimate_amount'))
)

display(df.select('estimate_id', 'estimate_amount').where(col('estimate_amount').isNull()))

In [0]:
# Check estimate types
#Checking for anamolies handling
df.select("estimate_type").distinct().show()

In [0]:
#standardize the data
df_cleaned = df \
    .filter(col('estimate_amount') > 0) \
    .withColumn('created_by', f.trim(col('created_by'))) \
    .withColumn('estimator_name', f.trim(col('estimator_name'))) \
    .withColumn('estimate_type', f.upper(f.trim(col('estimate_type'))))

print(f"Records after cleaning: {df_cleaned.count()}")

In [0]:
# Convert 'created_at' to date (yyyy-mm-dd) and update its type
df_cleaned = df_cleaned.withColumn('created_at', f.to_date(col('created_at')))

display(df_cleaned.select('estimate_id', 'created_at').limit(5))

In [0]:
df_cleaned = df_cleaned.withColumn('currency', f.lit('INR'))

display(df_cleaned.select('estimate_id', 'estimate_amount', 'currency').limit(5))

In [0]:
df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_estimate")

In [0]:
%sql
drop table if exists automobilerepair.silver.slv_estimate